In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import plotnine as gg
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [ ]:
adata = sc.read_h5ad("/workspace/data/250516_TF_perturbseq/250516_TF_perturbseq.annotated.h5ad")
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
adata.obs["consensus_target"].value_counts()

In [ ]:
gene_kos = adata.obs["consensus_target"].unique()
gene_kos = [g for g in gene_kos if g != "nontargeting"]
gene_kos = [g for g in gene_kos if g in adata.var_names]
np.random.seed(0)
heldout_perturbations = np.random.choice(gene_kos, size=10, replace=False)
training_perturbations = np.setdiff1d(gene_kos, heldout_perturbations)

In [ ]:
heldout_perturbations

In [ ]:
adata_control = adata[adata.obs["consensus_target"] == "nontargeting"].copy()
X_ctrl = adata_control.X

effects = []
for gene in heldout_perturbations:
    X_treated = adata[adata.obs["consensus_target"] == gene].X
    delta = X_treated.mean(axis=0) - X_ctrl.mean(axis=0)
    effects.append(delta.A1)
effects = np.array(effects)
print(effects.shape)

In [ ]:
effects.shape

In [ ]:
gene_idx = np.where(adata.var_names == "lacZ")[0][0]
adata[:, gene_idx].layers["counts"].max()

In [ ]:
effects.max(1)

In [ ]:
print("perturbed expressions")


print("delta / mean(ctrl)")
print(effects[:, gene_idx] / adata[:, gene_idx].X.mean())

In [ ]:
# lacZ regression

In [ ]:
gene_names = [
    "lacI",
    "crp",
    "cyaA",
    "ptsG",
]
gene_indices = np.where(adata.var_names.isin(gene_names))[0]
gene_indices

In [ ]:
adata.X.mean(axis=0).toarray()

In [ ]:
mean_expressions = adata.X.toarray().mean(axis=0)
expr_std = adata.X.toarray().std(axis=0)
max_expressions = adata.X.toarray().max(axis=0)
n_cells_expressing_gene = (adata.X.toarray() > 0).sum(axis=0)

for gene_name, gene_idx in zip(gene_names, gene_indices):
    print(gene_name)
    print(f"mean expression: {mean_expressions[gene_idx]:.2f}")
    print(f"std expression: {expr_std[gene_idx]:.2f}")
    print(f"max expression: {max_expressions[gene_idx]:.2f}")
    print(f"n cells expressing: {n_cells_expressing_gene[gene_idx]:.2f}")
    print()

print("--------------------------------")
print("averaged statistics")
print(f"mean expression: {mean_expressions.mean():.2f}")
print(f"std expression: {expr_std.mean():.2f}")
print(f"max expression: {max_expressions.mean():.2f}")
print(f"n cells expressing: {n_cells_expressing_gene.mean():.2f}")

In [ ]:
selected_genes = ["lacI", "crp", "ptsG"]


X_taylored = adata[:, selected_genes].X.toarray()
X_all = adata[:, adata.var_names != "lacZ"].X.toarray()
y = adata[:, "lacZ"].X.toarray().flatten()

X_taylored.shape, y.shape

In [ ]:
# train_indices, test_indices = train_test_split(
#     np.arange(X_all.shape[0]), test_size=0.2, random_state=0
# )
test_indices = (adata.obs["consensus_target"] == "crp").values
train_indices = np.setdiff1d(np.arange(X_all.shape[0]), test_indices)

X_train = X_all[train_indices]
X_test = X_all[test_indices]
X_taylored_train = X_taylored[train_indices]
X_taylored_test = X_taylored[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]

In [ ]:
plt.scatter(adata[:, "crp"].X.toarray().flatten(), adata[:, "lacZ"].X.toarray().flatten())

In [ ]:
lm_all = LinearRegression()
lm_all.fit(X_all[train_indices], y[train_indices])
lm_taylored = LinearRegression()
lm_taylored.fit(X_taylored[train_indices], y[train_indices])

In [ ]:
y_all_test = lm_all.predict(X_all[test_indices])
y_taylored_test = lm_taylored.predict(X_taylored[test_indices])

print("coefficients for ", selected_genes)
print(lm_taylored.coef_)
print("intercept")
print(lm_taylored.intercept_)

from scipy.stats import pearsonr

print("R2 using all features", pearsonr(y_all_test, y_test).statistic)
print("R2 using tailored features: ", pearsonr(y_taylored_test, y_test).statistic)

In [ ]:
pearsonr(y_all_test, y_test).statistic

In [ ]:
new_features = ["lacY", "galE", "hns"]
gene_indices = np.where(adata.var_names.isin(new_features))[0]

In [ ]:
for gene_name, gene_idx in zip(new_features, gene_indices):
    print(gene_name)
    print(f"mean expression: {mean_expressions[gene_idx]:.2f}")
    print(f"std expression: {expr_std[gene_idx]:.2f}")
    print(f"max expression: {max_expressions[gene_idx]:.2f}")
    print(f"n cells expressing: {n_cells_expressing_gene[gene_idx]:.2f}")
    print()

print("--------------------------------")
print("averaged statistics")
print(f"mean expression: {mean_expressions.mean():.2f}")
print(f"std expression: {expr_std.mean():.2f}")
print(f"max expression: {max_expressions.mean():.2f}")
print(f"n cells expressing: {n_cells_expressing_gene.mean():.2f}")

In [ ]:
selected_genes = ["lacY", "galE"]


X_taylored = adata[:, selected_genes].X.toarray()
X_all = adata[:, adata.var_names != "lacZ"].X.toarray()
y = adata[:, "lacZ"].X.toarray().flatten()

X_taylored.shape, y.shape

In [ ]:
# train_indices, test_indices = train_test_split(
#     np.arange(X_all.shape[0]), test_size=0.2, random_state=0
# )
test_indices = (adata.obs["consensus_target"] == "crp").values
train_indices = np.setdiff1d(np.arange(X_all.shape[0]), test_indices)

X_train = X_all[train_indices]
X_test = X_all[test_indices]
X_taylored_train = X_taylored[train_indices]
X_taylored_test = X_taylored[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]

In [ ]:
lm_all = LinearRegression()
lm_all.fit(X_all[train_indices], y[train_indices])
lm_taylored = LinearRegression()
lm_taylored.fit(X_taylored[train_indices], y[train_indices])

In [ ]:
y_all_test = lm_all.predict(X_all[test_indices])
y_taylored_test = lm_taylored.predict(X_taylored[test_indices])

print("coefficients for ", selected_genes)
print(lm_taylored.coef_)
print("intercept")
print(lm_taylored.intercept_)

from scipy.stats import pearsonr

print("R2 using all features", pearsonr(y_all_test, y_test).statistic)
print("R2 using tailored features: ", pearsonr(y_taylored_test, y_test).statistic)